# Lab 0: Python, numpy and matplotlib

This lab is for students with little experience of Python or numpy. It goes through the
numpy, matplotlib and class features used in all the labs of the course. Students who are
comfortable with them go directly to Lab 1.

A notebook is a sequence of cells. Select a cell and press `Shift+Enter` to run it. Run the
cells in order, from the top. Replace the lines `...  # TODO` with your code. Each
implementation is followed by a cell of automatic tests, which must print "all tests
passed".

Section 1 checks the installation described in the setup page (`SETUP.pdf`).

## 1. Installation check

The next cell must run without error and print the version of each package.

If an import fails, the notebook is not using the course environment: see the last section
of the setup page (`SETUP.pdf`).

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import seqlearn
from seqlearn import Agent, BernoulliBandit, checks, run
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()
print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("matplotlib", matplotlib.__version__)
print("seqlearn  ", seqlearn.__file__)

## 2. Arrays

A numpy array holds numbers of the same type. Operations apply to all its entries at once,
without a Python loop. Indices start at 0.

In [ ]:
means = np.array([0.5, 0.6, 0.4])
print("shape        :", means.shape)
print("entry 0      :", means[0])
print("times two    :", 2 * means)
print("plus 0.1     :", means + 0.1)
print("max, argmax  :", means.max(), np.argmax(means))
print("sum, mean    :", means.sum(), means.mean())
print("larger than 0.45:", means > 0.45)

An array can be indexed by an array of integers: the result has one entry per index. This
is used in every lab, to read a quantity attached to the arm played at each round.

In [ ]:
arms = np.array([0, 2, 2, 1, 2])  # arm played at rounds 1 to 5
print("mean of the arm played at each round:", means[arms])
print("rounds where arm 2 is played        :", arms == 2)
print("number of such rounds               :", (arms == 2).sum())

**Question 1.** Write `gaps(means)`, which returns the array of the gaps
$\Delta_a = \max_b \mu_b - \mu_a$.

In [ ]:
def gaps(means):
    ...  # TODO


checks.check_gaps(gaps)

**Question 2.** Write `counts(arms, n_arms)`, which returns the array of length `n_arms`
whose entry $a$ is the number of rounds where arm $a$ is played.

*Hint:* `np.bincount(..., minlength=...)`, or a loop over the arms with `(arms == a).sum()`.

In [ ]:
def counts(arms, n_arms):
    ...  # TODO


checks.check_counts(counts)

An experiment repeated `n_runs` times over `T` rounds is stored in an array of shape
`(n_runs, T)`: one row per run, one column per round. The argument `axis` selects the
direction of an operation: `axis=0` goes across runs, `axis=1` (or `axis=-1`, the last
axis) goes along time.

In [ ]:
x = np.array([[1.0, 0.0, 1.0, 1.0],
              [0.0, 0.0, 1.0, 0.0]])  # 2 runs, 4 rounds
print("shape                  :", x.shape)
print("mean over everything   :", x.mean())
print("mean across runs       :", x.mean(axis=0))
print("mean along time        :", x.mean(axis=1))
print("cumulative sum in time :")
print(np.cumsum(x, axis=-1))

**Question 3.** Write `running_mean(x)`, which returns the array of the same shape as `x`
whose entry at round $t$ is the mean of the first $t$ entries of `x` along the last axis:
$\frac{1}{t}\sum_{s=1}^t x_s$. The function must work for `x` of shape `(T,)` and
`(n_runs, T)`.

*Hint:* divide `np.cumsum(x, axis=-1)` by `np.arange(1, T + 1)`, where `T = x.shape[-1]`.

In [ ]:
def running_mean(x):
    ...  # TODO


checks.check_running_mean(running_mean)

## 3. Random numbers

All random draws go through a generator created with `np.random.default_rng(seed)`. Two
generators with the same seed produce the same draws, which makes an experiment
reproducible.

In [ ]:
rng = np.random.default_rng(0)
print("3 uniform draws on [0, 1):", rng.random(3))
print("3 integers in {0, 1, 2}  :", rng.integers(3, size=3))
print("array of shape (2, 4)    :")
print(rng.random((2, 4)))

rng_a, rng_b = np.random.default_rng(42), np.random.default_rng(42)
print("same seed, same draws    :", rng_a.random(2), rng_b.random(2))

**Question 4.** Write `bernoulli(p, size, rng)`, which returns an array of shape `size` of
independent draws equal to `1.0` with probability `p` and to `0.0` otherwise.

*Hint:* if $U$ is uniform on $[0, 1)$, then $\mathbb{P}(U < p) = p$. A boolean array `b` is
converted to floats with `b.astype(float)`.

In [ ]:
def bernoulli(p, size, rng):
    ...  # TODO


checks.check_bernoulli(bernoulli)

## 4. Figures

A figure is created with `fig, ax = plt.subplots()`, then drawn on with the methods of `ax`:
`ax.plot(x, y, label=...)` for a curve, `ax.axhline(y)` for a horizontal line,
`ax.fill_between(x, low, high, alpha=...)` for a band, `ax.set(xlabel=..., ylabel=...,
title=...)` for the labels and `ax.legend()` for the legend.

**Question 5.** Draw 5 independent runs of $T = 1000$ Bernoulli rewards of mean $p = 0.6$
(an array of shape `(5, 1000)`), with the seed 0. On a single figure, plot the running mean
of each run against the round $t = 1, \ldots, T$, and the horizontal line of height $p$.
Label the axes.

In [ ]:
P, T = 0.6, 1000
t = np.arange(1, T + 1)

...  # TODO

**Question 6.** Draw 200 runs instead of 5. Compute, for each round, the mean across runs
of the running mean, and the half-width $2\hat\sigma/\sqrt{n_{\text{runs}}}$ of its
uncertainty band, where $\hat\sigma$ is the standard deviation across runs
(`.std(axis=0, ddof=1)`). Plot the mean, the band (with `ax.fill_between`) and the line of
height $p$, for $t \ge 10$.

In [ ]:
N_RUNS = 200

...  # TODO

## 5. Classes

A class groups data (attributes) and functions acting on them (methods). The method
`__init__` is called when an object is created, and `self` is the object itself.

In [ ]:
class Counter:
    def __init__(self, start=0):
        self.value = start

    def increment(self):
        self.value += 1


counter = Counter()
counter.increment()
counter.increment()
print("value:", counter.value)

**Question 7.** Complete the class `RunningMean`, which keeps the mean of the numbers seen
so far without storing them. It has two attributes, `n` (number of observations) and
`value` (their mean, $0$ before any observation). The method `update(x)` applies the
incremental update of the lecture,
$$\hat\mu_{n+1} = \hat\mu_n + \frac{1}{n+1}\big(x_{n+1} - \hat\mu_n\big).$$

In [ ]:
class RunningMean:
    def __init__(self):
        self.n = 0
        self.value = 0.0

    def update(self, x):
        ...  # TODO


checks.check_running_mean_class(RunningMean)

A class can inherit from another one: it gets all its attributes and methods, and
redefines some of them. In the labs, every agent inherits from `seqlearn.Agent`, which
keeps track of the number of rounds played `self.t`, of the number of arms `self.n_arms`
and of the empirical means, and only redefines the method `act()`, which returns the arm
to play at the next round.

**Question 8.** Complete `RoundRobinAgent`, which plays the arms in turn:
$0, 1, \ldots, K-1, 0, 1, \ldots$ The method must return a Python integer.

*Hint:* `a % b` is the remainder of the division of `a` by `b`, and `self.t` is the number
of rounds already played.

In [ ]:
class RoundRobinAgent(Agent):
    def act(self):
        ...  # TODO


checks.check_round_robin(RoundRobinAgent)

The agent can now play against an environment. `run(env, agent, horizon)` returns the arms
played and the rewards received.

In [ ]:
env = BernoulliBandit([0.5, 0.6, 0.4], rng=0)
agent = RoundRobinAgent(env.n_arms)
arms, rewards = run(env, agent, horizon=9)
print("arms played    :", arms)
print("rewards        :", rewards)
print("pulls per arm  :", counts(arms, env.n_arms))
print("empirical means:", agent.mu_hat)

Continue with Labs 1 and 2 (`lab01-02.ipynb`).